# Zero-shot ensemble weight optimization

This notebook learns global ensemble weights on UBE4B, GRB2, PTEN activity/abundance and Alpha-Amylase, then applies the learned weights to blind DLG4 abundance and binding datasets.

Corrected VESPA and ProSST predictions are re-merged before optimization so that stale columns in older merged files are overwritten.

In [ ]:
# Imports
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution, minimize
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import ndcg_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [ ]:
# Paths
BASE_DIR = Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase")
MERGED_DATA_DIR = BASE_DIR / "BetaTest"
FINAL_DIR = BASE_DIR / "Final"

VESPA_OUTPUT_DIR = FINAL_DIR / "VESPA" / "vespa_zero_shot_outputs"
PROSST_PREDICTION_DIR = FINAL_DIR / "ProSST" / "predictions"

# "Training" datasets
TRAIN_DATASETS = {
    "UBE4B": {
        "path": MERGED_DATA_DIR / "UBE4B_merged.csv",
        "target_cols": ["score"],
    },
    "GRB2": {
        "path": MERGED_DATA_DIR / "GRB2_merged.csv",
        "target_cols": ["score"],
    },
    "PTEN_activity": {
        "path": MERGED_DATA_DIR / "PTEN_activity_merged.csv",
        "target_cols": ["score"],
    },
    "PTEN_abundance": {
        "path": MERGED_DATA_DIR / "PTEN_abundance_merged.csv",
        "target_cols": ["score"],
    },
    "Alpha-Amylase": {
        "path": MERGED_DATA_DIR / "Alpha-Amylase_merged.csv",
        "target_cols": ["expression", "thermostability", "specific activity"],
    },
}

# Corrected VESPA outputs from the final VESPA pipeline
VESPA_FILE_MAP = {
    "UBE4B": VESPA_OUTPUT_DIR / "UBE4B_with_vespa.csv",
    "GRB2": VESPA_OUTPUT_DIR / "GRB2_with_vespa.csv",
    "PTEN_activity": VESPA_OUTPUT_DIR / "PTEN_activity_with_vespa.csv",
    "PTEN_abundance": VESPA_OUTPUT_DIR / "PTEN_abundance_with_vespa.csv",
    "Alpha-Amylase": VESPA_OUTPUT_DIR / "Alpha-Amylase_with_vespa.csv",
}

# ProSST predictions from the final ProSST pipeline
PROSST_FILE_MAP = {
    "UBE4B": None,
    "GRB2": None,
    "PTEN_activity": None,
    "PTEN_abundance": None,
    "Alpha-Amylase": None,
    "DLG4_abundance": None,
    "DLG4_binding": None,
}

OUTPUT_DIR = FINAL_DIR / "ensemble_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Raw score columns
MODEL_COLS = [
    "ESM2",
    "ddg",
    "VESPAl",
    "prosst",
    "esm3_index",
    "esmIF_index",
    "eve_index",
]

# Models with this sign are converted to a unified direction where higher = better
DEFAULT_MODEL_SIGN = {
    "ESM2": 1,
    "ddg": 1,
    "VESPAl": -1,
    "prosst": 1,
    "esm3_index": 1,
    "esmIF_index": 1,
    "eve_index": 1,
}

OPTIMIZATION_FRACS = [0.05, 0.10, 0.20]
RANDOM_SEED = 42

In [ ]:
def read_table(path, sep=None):
    path = Path(path)
    if sep is not None:
        return pd.read_csv(path, sep=sep)
    if path.suffix.lower() in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="	")
    return pd.read_csv(path)


def numeric_array(series):
    return pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)


def make_relevance(values):
    """Shift experimental values to non-negative relevance scores for sklearn NDCG."""
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return values
    values = values - np.nanmin(values)
    values = np.clip(values, 0.0, None)
    return values


def ndcg_at_fraction(true_values, pred_scores, frac=0.10):
    true_values = np.asarray(true_values, dtype=float)
    pred_scores = np.asarray(pred_scores, dtype=float)
    mask = np.isfinite(true_values) & np.isfinite(pred_scores)
    true_values = true_values[mask]
    pred_scores = pred_scores[mask]

    if len(true_values) < 3:
        return np.nan

    y_true = make_relevance(true_values)
    if np.nanmax(y_true) == 0:
        return np.nan

    k = max(1, min(int(np.ceil(frac * len(y_true))), len(y_true)))
    return float(ndcg_score(y_true.reshape(1, -1), pred_scores.reshape(1, -1), k=k))


def summarize_prediction(true_values, pred_scores, fracs=(0.05, 0.10, 0.20)):
    true_values = np.asarray(true_values, dtype=float)
    pred_scores = np.asarray(pred_scores, dtype=float)
    mask = np.isfinite(true_values) & np.isfinite(pred_scores)
    out = {"n": int(mask.sum())}

    if mask.sum() < 3:
        out.update({"pearson": np.nan, "spearman": np.nan})
        for frac in fracs:
            out[f"ndcg@{int(frac*100)}%"] = np.nan
        return out

    y = true_values[mask]
    p = pred_scores[mask]
    out["pearson"] = float(pearsonr(y, p).statistic)
    out["spearman"] = float(spearmanr(y, p).correlation)
    for frac in fracs:
        out[f"ndcg@{int(frac*100)}%"] = ndcg_at_fraction(y, p, frac=frac)
    return out


In [ ]:
# Rank and orientation helpers
def infer_model_sign(df, model_col, target_cols):
    """Infer whether a raw model score should be multiplied by +1 or -1.

    The sign is chosen so that the average Spearman correlation with available training targets is positive.
    """
    if model_col not in df.columns:
        return np.nan

    x = numeric_array(df[model_col])
    corrs = []
    for target_col in target_cols:
        if target_col not in df.columns:
            continue
        y = numeric_array(df[target_col])
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() >= 3:
            corr = spearmanr(y[mask], x[mask]).correlation
            if np.isfinite(corr):
                corrs.append(corr)

    if not corrs:
        return DEFAULT_MODEL_SIGN.get(model_col, 1)

    return 1 if np.nanmean(corrs) >= 0 else -1


def infer_global_model_signs(tasks, model_cols):
    rows = []
    for task in tasks:
        df = task["df"]
        target_col = task["target_col"]
        for model_col in model_cols:
            if model_col not in df.columns:
                continue
            x = numeric_array(df[model_col])
            y = numeric_array(df[target_col])
            mask = np.isfinite(x) & np.isfinite(y)
            if mask.sum() >= 3:
                corr = spearmanr(y[mask], x[mask]).correlation
                if np.isfinite(corr):
                    rows.append({"task": task["task_name"], "model": model_col, "spearman_raw": corr})

    corr_df = pd.DataFrame(rows)
    signs = {}
    for model_col in model_cols:
        vals = corr_df.loc[corr_df["model"] == model_col, "spearman_raw"]
        if len(vals) == 0:
            signs[model_col] = DEFAULT_MODEL_SIGN.get(model_col, 1)
        else:
            signs[model_col] = 1 if vals.mean() >= 0 else -1
    return signs, corr_df


def add_oriented_model_columns(df, model_cols, model_signs):
    df = df.copy()
    for model_col in model_cols:
        if model_col not in df.columns:
            continue
        df[f"{model_col}_oriented"] = numeric_array(df[model_col]) * model_signs.get(model_col, 1)
    return df


def add_rank_columns(df, model_cols, rank_suffix="_rank_norm"):
    """Add normalized ranks for oriented model columns.

    Convention: 0 = best, 1 = worst. Lower rank values are better.
    """
    df = df.copy()
    for model_col in model_cols:
        oriented_col = f"{model_col}_oriented"
        if oriented_col not in df.columns:
            continue
        x = pd.to_numeric(df[oriented_col], errors="coerce")
        valid = x.notna() & np.isfinite(x)
        n = int(valid.sum())
        out_col = f"{model_col}{rank_suffix}"
        df[out_col] = np.nan
        if n == 0:
            continue
        ranks = x[valid].rank(method="average", ascending=False)
        df.loc[valid, out_col] = 0.0 if n == 1 else (ranks - 1) / (n - 1)
    return df


def available_models_for_df(df, model_cols):
    return [c for c in model_cols if c in df.columns]


In [9]:
# Load training data and construct one task per target column
training_dfs = {}
training_tasks = []

for dataset_name, cfg in TRAIN_DATASETS.items():
    df = read_table(cfg["path"], sep=cfg.get("sep"))
    training_dfs[dataset_name] = df

    available_models = available_models_for_df(df, MODEL_COLS)
    print(f"{dataset_name}: rows={len(df)}, available_models={available_models}")

    for target_col in cfg["target_cols"]:
        if target_col not in df.columns:
            print(f"  [WARN] missing target column: {target_col}")
            continue
        training_tasks.append({
            "dataset_name": dataset_name,
            "target_col": target_col,
            "task_name": f"{dataset_name}__{target_col}",
            "df": df,
        })

print(f"\nNumber of optimization tasks: {len(training_tasks)}")
print([task["task_name"] for task in training_tasks])


UBE4B: rows=940, available_models=['ESM2', 'ddg', 'VESPAl', 'prosst', 'esm3_index', 'esmIF_index', 'eve_index']
GRB2: rows=1034, available_models=['ESM2', 'ddg', 'VESPAl', 'prosst', 'esm3_index', 'esmIF_index', 'eve_index']
PTEN_activity: rows=6564, available_models=['ESM2', 'ddg', 'VESPAl', 'prosst', 'esm3_index', 'esmIF_index', 'eve_index']
PTEN_abundance: rows=4387, available_models=['ESM2', 'ddg', 'VESPAl', 'prosst', 'esm3_index', 'esmIF_index', 'eve_index']
Alpha-Amylase: rows=8075, available_models=['ESM2', 'ddg', 'VESPAl', 'prosst', 'esm3_index', 'esmIF_index', 'eve_index']

Number of optimization tasks: 7
['UBE4B__score', 'GRB2__score', 'PTEN_activity__score', 'PTEN_abundance__score', 'Alpha-Amylase__expression', 'Alpha-Amylase__thermostability', 'Alpha-Amylase__specific activity']


In [ ]:
# Overwrite stale VESPA and ProSST columns with corrected prediction files
VARIANT_LIKE_COLS = ["variant", "mutant"]


def normalize_dataset_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def standardize_variant_column(df: pd.DataFrame) -> pd.DataFrame:
    """Create a normalized `variant` column for robust merging."""
    df = df.copy()
    if "variant" not in df.columns:
        for col in VARIANT_LIKE_COLS[1:]:
            if col in df.columns:
                df = df.rename(columns={col: "variant"})
                break
    if "variant" not in df.columns:
        raise KeyError(f"No variant-like column found. Columns: {df.columns.tolist()}")
    df = df.loc[df["variant"].notna()].copy()
    df["variant"] = df["variant"].astype(str).str.strip().str.upper()
    return df


def read_prediction_file(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t")
    return pd.read_csv(path)


def find_prediction_file(prediction_dir: Path, dataset_name: str, explicit_path: Path | None = None) -> Path | None:
    """Find a prediction CSV/TSV by dataset name if no explicit file is configured."""
    if explicit_path is not None:
        explicit_path = Path(explicit_path)
        return explicit_path if explicit_path.exists() else None

    prediction_dir = Path(prediction_dir)
    if not prediction_dir.exists():
        return None

    wanted = normalize_dataset_name(dataset_name)
    aliases = {
        "ptenactivity": ["ptenactivity", "ptenactivity", "activity"],
        "ptenabundance": ["ptenabundance", "abundance"],
        "alphaamylase": ["alphaamylase", "amylase"],
        "dlg4abundance": ["dlg4abundance", "dlg4", "abundance"],
        "dlg4binding": ["dlg4binding", "dlg4", "binding"],
    }
    search_terms = aliases.get(wanted, [wanted])

    candidates = []
    for suffix in ["*.csv", "*.tsv", "*.txt"]:
        candidates.extend(prediction_dir.rglob(suffix))

    scored = []
    for path in candidates:
        normalized = normalize_dataset_name(path.stem)
        score = sum(term in normalized for term in search_terms)
        if wanted in normalized:
            score += 3
        if score > 0:
            scored.append((score, path.stat().st_mtime, path))

    if not scored:
        return None

    scored.sort(reverse=True)
    return scored[0][2]


def infer_prediction_column(df_pred: pd.DataFrame, preferred_cols: list[str], output_col: str) -> str:
    """Pick the score column from a prediction file."""
    for col in preferred_cols:
        if col in df_pred.columns:
            return col

    numeric_cols = []
    skip = set(VARIANT_LIKE_COLS + ["sequence", "mutated_sequence", "protein_id", "id", "ID"])
    for col in df_pred.columns:
        if col in skip:
            continue
        values = pd.to_numeric(df_pred[col], errors="coerce")
        if values.notna().sum() > 0:
            numeric_cols.append(col)

    if len(numeric_cols) == 1:
        return numeric_cols[0]

    raise KeyError(
        f"Could not infer prediction column for {output_col}. "
        f"Preferred columns: {preferred_cols}. Available columns: {df_pred.columns.tolist()}"
    )


def overwrite_single_prediction_column(
    df_base: pd.DataFrame,
    prediction_path: Path,
    output_col: str,
    preferred_cols: list[str],
    dataset_name: str,
) -> pd.DataFrame:
    """Overwrite one model column in df_base using a prediction file."""
    df_base = standardize_variant_column(df_base)
    df_pred = standardize_variant_column(read_prediction_file(prediction_path))
    pred_col = infer_prediction_column(df_pred, preferred_cols=preferred_cols, output_col=output_col)

    df_pred_small = (
        df_pred[["variant", pred_col]]
        .copy()
        .drop_duplicates("variant")
        .rename(columns={pred_col: output_col})
    )
    df_pred_small[output_col] = pd.to_numeric(df_pred_small[output_col], errors="coerce")

    df_base = df_base.drop(columns=[output_col], errors="ignore")
    merged = df_base.merge(df_pred_small, on="variant", how="left")

    missing = merged[output_col].isna().sum()
    print(
        f"[{dataset_name}] overwritten {output_col:8s} from {prediction_path.name} "
        f"using column '{pred_col}' | NaNs: {missing}/{len(merged)}"
    )
    return merged


def overwrite_corrected_predictions(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    """Overwrite corrected VESPA and ProSST predictions where files are available."""
    df = df.copy()

    vespa_path = VESPA_FILE_MAP.get(dataset_name)
    if vespa_path is not None and Path(vespa_path).exists():
        df = overwrite_single_prediction_column(
            df_base=df,
            prediction_path=Path(vespa_path),
            output_col="VESPAl",
            preferred_cols=["VESPAl", "vespal", "VESPA", "VESPA_score"],
            dataset_name=dataset_name,
        )
    else:
        print(f"[{dataset_name}] no corrected VESPA file found; keeping existing VESPAl if present.")

    prosst_path = find_prediction_file(
        prediction_dir=PROSST_PREDICTION_DIR,
        dataset_name=dataset_name,
        explicit_path=PROSST_FILE_MAP.get(dataset_name),
    )
    if prosst_path is not None:
        df = overwrite_single_prediction_column(
            df_base=df,
            prediction_path=prosst_path,
            output_col="prosst",
            preferred_cols=["prosst", "ProSST", "ProSST_score", "prosst_score", "score"],
            dataset_name=dataset_name,
        )
    else:
        print(f"[{dataset_name}] no corrected ProSST file found; keeping existing prosst if present.")

    return df


# Apply overwrites to training datasets and refresh task references.
training_dfs = {
    dataset_name: overwrite_corrected_predictions(df, dataset_name)
    for dataset_name, df in training_dfs.items()
}

for task in training_tasks:
    task["df"] = training_dfs[task["dataset_name"]]

print("\nAvailable model columns after overwriting corrected predictions:")
for dataset_name, df in training_dfs.items():
    print(f"{dataset_name:16s}: {available_models_for_df(df, MODEL_COLS)}")


[UBE4B] overwritten VESPAl   from UBE4B_with_vespa.csv using column 'VESPAl' | NaNs: 0/940
[UBE4B] overwritten prosst   from UBE4B_prosst_multi.tsv using column 'prosst' | NaNs: 0/940
[GRB2] overwritten VESPAl   from GRB2_with_vespa.csv using column 'VESPAl' | NaNs: 0/1034
[GRB2] overwritten prosst   from GRB2_prosst_multi.tsv using column 'prosst' | NaNs: 0/1034
[PTEN_activity] overwritten VESPAl   from PTEN_activity_with_vespa.csv using column 'VESPAl' | NaNs: 0/6564
[PTEN_activity] overwritten prosst   from PTEN_activity_prosst.tsv using column 'prosst' | NaNs: 0/6564
[PTEN_abundance] overwritten VESPAl   from PTEN_abundance_with_vespa.csv using column 'VESPAl' | NaNs: 0/4387
[PTEN_abundance] overwritten prosst   from PTEN_abundance_prosst.tsv using column 'prosst' | NaNs: 0/4387
[Alpha-Amylase] overwritten VESPAl   from Alpha-Amylase_with_vespa.csv using column 'VESPAl' | NaNs: 564/8075
[Alpha-Amylase] overwritten prosst   from Alpha_Amylase_prosst.csv using column 'prosst' | NaNs:

In [ ]:
MODEL_SIGNS, raw_corr_df = infer_global_model_signs(training_tasks, MODEL_COLS)

print("Model signs used for unified direction where higher = better:")
for model, sign in MODEL_SIGNS.items():
    print(f"  {model:12s}: {sign:+d}")

display(raw_corr_df.pivot_table(index="model", values="spearman_raw", aggfunc=["mean", "count"]).round(3))


Model signs used for unified direction where higher = better:
  ESM2        : +1
  ddg         : +1
  VESPAl      : -1
  prosst      : +1
  esm3_index  : +1
  esmIF_index : +1
  eve_index   : +1


,mean,count
,spearman_raw,spearman_raw
model,,
ESM2,0.518,7
VESPAl,-0.458,7
ddg,0.325,7
esm3_index,0.466,7
esmIF_index,0.458,7
eve_index,0.276,7
prosst,0.444,7


In [ ]:
training_dfs_prepared = {}

for dataset_name, df in training_dfs.items():
    df_prep = add_oriented_model_columns(df, MODEL_COLS, MODEL_SIGNS)
    df_prep = add_rank_columns(df_prep, MODEL_COLS)
    training_dfs_prepared[dataset_name] = df_prep

for task in training_tasks:
    task["df"] = training_dfs_prepared[task["dataset_name"]]

for dataset_name, df in training_dfs_prepared.items():
    rank_cols = [c for c in df.columns if c.endswith("_rank_norm")]
    print(dataset_name, rank_cols)


UBE4B ['ESM2_rank_norm', 'ddg_rank_norm', 'VESPAl_rank_norm', 'prosst_rank_norm', 'esm3_index_rank_norm', 'esmIF_index_rank_norm', 'eve_index_rank_norm']
GRB2 ['ESM2_rank_norm', 'ddg_rank_norm', 'VESPAl_rank_norm', 'prosst_rank_norm', 'esm3_index_rank_norm', 'esmIF_index_rank_norm', 'eve_index_rank_norm']
PTEN_activity ['ESM2_rank_norm', 'ddg_rank_norm', 'VESPAl_rank_norm', 'prosst_rank_norm', 'esm3_index_rank_norm', 'esmIF_index_rank_norm', 'eve_index_rank_norm']
PTEN_abundance ['ESM2_rank_norm', 'ddg_rank_norm', 'VESPAl_rank_norm', 'prosst_rank_norm', 'esm3_index_rank_norm', 'esmIF_index_rank_norm', 'eve_index_rank_norm']
Alpha-Amylase ['ESM2_rank_norm', 'ddg_rank_norm', 'VESPAl_rank_norm', 'prosst_rank_norm', 'esm3_index_rank_norm', 'esmIF_index_rank_norm', 'eve_index_rank_norm']


In [ ]:
# Baseline model performance on training tasks
baseline_rows = []

for task in training_tasks:
    df = task["df"]
    target_col = task["target_col"]
    for model_col in MODEL_COLS:
        score_col = f"{model_col}_oriented"
        if score_col not in df.columns:
            continue
        metrics = summarize_prediction(numeric_array(df[target_col]), numeric_array(df[score_col]), fracs=OPTIMIZATION_FRACS)
        baseline_rows.append({
            "task": task["task_name"],
            "dataset": task["dataset_name"],
            "target": target_col,
            "model": model_col,
            **metrics,
        })

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df.sort_values(["task", "ndcg@10%"], ascending=[True, False]).round(3))

baseline_df.to_csv(OUTPUT_DIR / "baseline_model_performance.csv", index=False)

,task,dataset,target,model,n,pearson,spearman,ndcg@5%,ndcg@10%,ndcg@20%
30,Alpha-Amylase__expression,Alpha-Amylase,expression,VESPAl,7511,0.471,0.496,0.682,0.725,0.760
34,Alpha-Amylase__expression,Alpha-Amylase,expression,eve_index,7343,0.455,0.473,0.648,0.697,0.739
28,Alpha-Amylase__expression,Alpha-Amylase,expression,ESM2,7575,0.531,0.541,0.642,0.693,0.743
33,Alpha-Amylase__expression,Alpha-Amylase,expression,esmIF_index,7575,0.472,0.486,0.640,0.693,0.741
32,Alpha-Amylase__expression,Alpha-Amylase,expression,esm3_index,7575,0.431,0.440,0.604,0.648,0.700
31,Alpha-Amylase__expression,Alpha-Amylase,expression,prosst,7575,0.402,0.373,0.460,0.530,0.599
29,Alpha-Amylase__expression,Alpha-Amylase,expression,ddg,7396,0.222,0.421,0.439,0.528,0.628
44,Alpha-Amylase__specific activity,Alpha-Amylase,specific activity,VESPAl,7511,0.444,0.387,0.524,0.599,0.677
48,Alpha-Amylase__specific activity,Alpha-Amylase,specific activity,eve_index,7343,0.317,0.359,0.329,0.442,0.553
42,Alpha-Amylase__specific activity,Alpha-Amylase,specific activity,ESM2,7575,0.342,0.387,0.334,0.437,0.549


In [ ]:
# Build optimization matrices
# Each model rank has convention 0=best, 1=worst
# For NDCG, we use ensemble_score = -ensemble_rank so that higher = better

ACTIVE_MODELS = [m for m in MODEL_COLS if any(m in task["df"].columns for task in training_tasks)]
RANK_COLS = [f"{m}_rank_norm" for m in ACTIVE_MODELS]

prepared_tasks = []
for task in training_tasks:
    df = task["df"]
    target_col = task["target_col"]

    missing = [c for c in RANK_COLS if c not in df.columns]
    if missing:
        print(f"[WARN] {task['task_name']} missing rank columns: {missing}")

    cols_here = [c for c in RANK_COLS if c in df.columns]
    X = df[cols_here].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    y = numeric_array(df[target_col])
    mask = np.isfinite(y) & np.all(np.isfinite(X), axis=1)

    if mask.sum() < 3:
        print(f"[WARN] Skipping {task['task_name']}: only {mask.sum()} complete rows")
        continue

    # Re-expand X to all active models; missing columns would be NaN, but we skip tasks missing columns by default
    X_full = df[RANK_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    mask = np.isfinite(y) & np.all(np.isfinite(X_full), axis=1)

    prepared_tasks.append({
        "task_name": task["task_name"],
        "dataset_name": task["dataset_name"],
        "target_col": target_col,
        "X": X_full[mask],
        "y": y[mask],
        "n": int(mask.sum()),
    })

print("Active models:", ACTIVE_MODELS)
print("Prepared tasks:")
for task in prepared_tasks:
    print(f"  {task['task_name']:40s} n={task['n']}")


Active models: ['ESM2', 'ddg', 'VESPAl', 'prosst', 'esm3_index', 'esmIF_index', 'eve_index']
Prepared tasks:
  UBE4B__score                             n=930
  GRB2__score                              n=1034
  PTEN_activity__score                     n=6547
  PTEN_abundance__score                    n=4368
  Alpha-Amylase__expression                n=7138
  Alpha-Amylase__thermostability           n=7138
  Alpha-Amylase__specific activity         n=7138


In [ ]:
# Objective and optimization

def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    w = np.exp(z)
    return w / w.sum()


def evaluate_weights(weights, tasks, fracs=OPTIMIZATION_FRACS):
    weights = np.asarray(weights, dtype=float)
    rows = []
    values = []

    for task in tasks:
        ensemble_rank = task["X"] @ weights
        ensemble_score = -ensemble_rank

        row = {
            "task": task["task_name"],
            "dataset": task["dataset_name"],
            "target": task["target_col"],
            "n": task["n"],
        }

        for frac in fracs:
            value = ndcg_at_fraction(task["y"], ensemble_score, frac=frac)
            row[f"ndcg@{int(frac*100)}%"] = value
            if np.isfinite(value):
                values.append(value)

        row["spearman"] = spearmanr(task["y"], ensemble_score).correlation
        rows.append(row)

    return pd.DataFrame(rows), float(np.nanmean(values))


def objective_logits(logits):
    weights = softmax(logits)
    _, mean_ndcg = evaluate_weights(weights, prepared_tasks, fracs=OPTIMIZATION_FRACS)
    if not np.isfinite(mean_ndcg):
        return 1e6
    return -mean_ndcg

bounds = [(-6, 6)] * len(ACTIVE_MODELS)

result_de = differential_evolution(
    objective_logits,
    bounds=bounds,
    seed=RANDOM_SEED,
    maxiter=1000,
    popsize=30,
    polish=False,
    workers=1,
    updating="immediate",
)

result_local = minimize(
    objective_logits,
    result_de.x,
    method="Nelder-Mead",
    options={"maxiter": 5000, "xatol": 1e-8, "fatol": 1e-8},
)

best_logits = result_local.x if result_local.fun <= result_de.fun else result_de.x
BEST_WEIGHTS = softmax(best_logits)

weights_df = pd.DataFrame({
    "model": ACTIVE_MODELS,
    "weight": BEST_WEIGHTS,
    "sign": [MODEL_SIGNS[m] for m in ACTIVE_MODELS],
})
weights_df = weights_df.sort_values("weight", ascending=False).reset_index(drop=True)

eval_df, mean_ndcg = evaluate_weights(BEST_WEIGHTS, prepared_tasks, fracs=OPTIMIZATION_FRACS)

print(f"Best mean NDCG across tasks/fracs: {mean_ndcg:.4f}")
display(weights_df)
display(eval_df.round(3))

weights_df.to_csv(OUTPUT_DIR / "optimized_zero_shot_weights.csv", index=False)
eval_df.to_csv(OUTPUT_DIR / "optimized_training_task_performance.csv", index=False)

with open(OUTPUT_DIR / "optimized_zero_shot_weights.json", "w") as f:
    json.dump({
        "active_models": ACTIVE_MODELS,
        "weights": dict(zip(ACTIVE_MODELS, BEST_WEIGHTS.tolist())),
        "model_signs": MODEL_SIGNS,
        "optimization_fracs": OPTIMIZATION_FRACS,
        "mean_ndcg": mean_ndcg,
    }, f, indent=2)

Best mean NDCG across tasks/fracs: 0.7766


,model,weight,sign
0,ESM2,0.274686,1
1,esmIF_index,0.250351,1
2,prosst,0.194959,1
3,esm3_index,0.173740,1
4,VESPAl,0.105549,-1
5,eve_index,0.000696,1
6,ddg,0.000019,1


,task,dataset,target,n,ndcg@5%,ndcg@10%,ndcg@20%,spearman
0,UBE4B__score,UBE4B,score,930,0.709,0.765,0.828,0.587
1,GRB2__score,GRB2,score,1034,0.913,0.922,0.930,0.773
2,PTEN_activity__score,PTEN_activity,score,6547,0.853,0.881,0.906,0.619
3,PTEN_abundance__score,PTEN_abundance,score,4368,0.789,0.814,0.851,0.556
4,Alpha-Amylase__expression,Alpha-Amylase,expression,7138,0.697,0.752,0.798,0.586
5,Alpha-Amylase__thermostability,Alpha-Amylase,thermostability,7138,0.653,0.710,0.742,0.405
6,Alpha-Amylase__specific activity,Alpha-Amylase,specific activity,7138,0.520,0.596,0.677,0.354


In [18]:
# Optional: compare optimized ensemble to equal-weight ensemble
EQUAL_WEIGHTS = np.ones(len(ACTIVE_MODELS), dtype=float) / len(ACTIVE_MODELS)

equal_eval_df, equal_mean_ndcg = evaluate_weights(EQUAL_WEIGHTS, prepared_tasks, fracs=OPTIMIZATION_FRACS)
opt_eval_df, opt_mean_ndcg = evaluate_weights(BEST_WEIGHTS, prepared_tasks, fracs=OPTIMIZATION_FRACS)

print(f"Equal-weight mean NDCG: {equal_mean_ndcg:.4f}")
print(f"Optimized mean NDCG:    {opt_mean_ndcg:.4f}")

display(
    opt_eval_df[["task", "ndcg@5%", "ndcg@10%", "ndcg@20%", "spearman"]]
    .rename(columns={"ndcg@5%": "opt_ndcg@5%", "ndcg@10%": "opt_ndcg@10%", "ndcg@20%": "opt_ndcg@20%", "spearman": "opt_spearman"})
    .merge(
        equal_eval_df[["task", "ndcg@5%", "ndcg@10%", "ndcg@20%", "spearman"]]
        .rename(columns={"ndcg@5%": "equal_ndcg@5%", "ndcg@10%": "equal_ndcg@10%", "ndcg@20%": "equal_ndcg@20%", "spearman": "equal_spearman"}),
        on="task",
        how="left",
    )
    .round(3)
)


Equal-weight mean NDCG: 0.7737
Optimized mean NDCG:    0.7766


,task,opt_ndcg@5%,opt_ndcg@10%,opt_ndcg@20%,opt_spearman,equal_ndcg@5%,equal_ndcg@10%,equal_ndcg@20%,equal_spearman
0,UBE4B__score,0.709,0.765,0.828,0.587,0.690,0.751,0.815,0.546
1,GRB2__score,0.913,0.922,0.930,0.773,0.911,0.915,0.926,0.756
2,PTEN_activity__score,0.853,0.881,0.906,0.619,0.849,0.876,0.906,0.601
3,PTEN_abundance__score,0.789,0.814,0.851,0.556,0.787,0.816,0.847,0.548
4,Alpha-Amylase__expression,0.697,0.752,0.798,0.586,0.701,0.752,0.800,0.606
5,Alpha-Amylase__thermostability,0.653,0.710,0.742,0.405,0.651,0.720,0.745,0.430
6,Alpha-Amylase__specific activity,0.520,0.596,0.677,0.354,0.522,0.591,0.675,0.376


In [ ]:
# Build blind DLG4 datasets from separate model output folders
# and apply optimized ensemble weights

# Paths

BASE_DIR = Path("/Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final")

OUTPUT_DIR = BASE_DIR / "ensemble_outputs" / "dlg4_blind"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BLIND_DATASETS = {
    "dlg4-2022-abundance": {
        "files": {
            "ESM2": {
                "path": BASE_DIR / "ESM-2" / "esm2_zero_shot_outputs" / "DLG4_abundance_with_esm2.csv",
                "pred_col": "ESM2",
            },
            "esm3_index": {
                "path": BASE_DIR / "ESM3_ESMIF_EVE" / "dlg4-2022-abundance_ESM3.tsv",
                "sep": "\t",
                "pred_col": "esm3_index",
            },
            "esmIF_index": {
                "path": BASE_DIR / "ESM3_ESMIF_EVE" / "dlg4-2022-abundance_ESMIF.tsv",
                "sep": "\t",
                "pred_col": "esmIF_index",
            },
            "eve_index": {
                "path": BASE_DIR / "ESM3_ESMIF_EVE" / "dlg4-2022-abundance_EVE.tsv",
                "sep": "\t",
                "pred_col": "EVE_score",
            },
            "prosst": {
                "path": BASE_DIR / "ProSST" / "predictions" / "DLG4_abundance_prosst.tsv",
                "sep": "\t",
                "pred_col": "prosst",
            },
            "VESPAl": {
                "path": BASE_DIR / "VESPA" / "vespa_zero_shot_outputs" / "DLG4_abundance_with_vespa.csv",
                "pred_col": "VESPA_score",
            },
        }
    },

    "dlg4-2022-binding": {
        "files": {
            "ESM2": {
                "path": BASE_DIR / "ESM-2" / "esm2_zero_shot_outputs" / "DLG4_binding_with_esm2.csv",
                "pred_col": "ESM2",
            },
            "esm3_index": {
                "path": BASE_DIR / "ESM3_ESMIF_EVE" / "dlg4-2022-binding_ESM3.tsv",
                "sep": "\t",
                "pred_col": "esm3_index",
            },
            "esmIF_index": {
                "path": BASE_DIR / "ESM3_ESMIF_EVE" / "dlg4-2022-binding_ESMIF.tsv",
                "sep": "\t",
                "pred_col": "esmIF_index",
            },
            "eve_index": {
                "path": BASE_DIR / "ESM3_ESMIF_EVE" / "dlg4-2022-binding_EVE.tsv",
                "sep": "\t",
                "pred_col": "EVE_score",
            },
            "prosst": {
                "path": BASE_DIR / "ProSST" / "predictions" / "DLG4_binding_prosst.tsv",
                "sep": "\t",
                "pred_col": "prosst",
            },
            "VESPAl": {
                "path": BASE_DIR / "VESPA" / "vespa_zero_shot_outputs" / "DLG4_binding_with_vespa.csv",
                "pred_col": "VESPA_score",
            },
        }
    },
}


# Helper functions
def read_table(path, sep=None):
    path = Path(path)

    if sep is None:
        if path.suffix.lower() == ".csv":
            sep = ","
        else:
            sep = "\t"

    return pd.read_csv(path, sep=sep)


def read_model_prediction_file(model_name, file_cfg):
    """
    Reads one model file and returns only:
    - variant
    - optional metadata columns
    - model prediction column renamed to model_name
    """

    path = Path(file_cfg["path"])
    sep = file_cfg.get("sep", None)
    pred_col = file_cfg["pred_col"]

    if not path.exists():
        raise FileNotFoundError(f"File not found for {model_name}: {path}")

    df = read_table(path, sep=sep)

    if "num_mutations" in df.columns:
        n_before = len(df)
        df["num_mutations"] = pd.to_numeric(df["num_mutations"], errors="coerce")
        df = df[df["num_mutations"] == 1].copy()
        n_after = len(df)
        print(f"{model_name}: filtered num_mutations == 1: {n_before} -> {n_after}")

    if "variant" not in df.columns:
        raise KeyError(
            f"'variant' column missing in {path.name}. "
            f"Available columns: {list(df.columns)}"
        )

    if pred_col not in df.columns:
        raise KeyError(
            f"Prediction column '{pred_col}' missing in {path.name}. "
            f"Available columns: {list(df.columns)}"
        )

    keep_cols = ["variant"]

    # Metadata / target columns, falls vorhanden
    optional_cols = [
        "num_mutations",
        "sequence",
        "score",
        "mutant",
        "mutated_sequence",
    ]

    for col in optional_cols:
        if col in df.columns and col not in keep_cols:
            keep_cols.append(col)

    keep_cols.append(pred_col)

    df = df[keep_cols].copy()

    if "score" in df.columns:
        df = df.rename(columns={"score": "experimental_score"})

    df = df.rename(columns={pred_col: model_name})

    if df["variant"].duplicated().any():
        duplicated = df.loc[df["variant"].duplicated(), "variant"].unique()[:10]
        print(f"[WARNING] Duplicate variants in {path.name}: {duplicated}")
        df = df.drop_duplicates(subset=["variant"], keep="first")

    return df


def merge_model_predictions(dataset_name, dataset_cfg, merge_how="outer"):
    """
    Reads all model prediction files for one dataset and merges them by variant.
    """

    merged = None

    for model_name, file_cfg in dataset_cfg["files"].items():
        df_model = read_model_prediction_file(model_name, file_cfg)

        print(f"{dataset_name} | {model_name}: loaded {df_model.shape}")

        if merged is None:
            merged = df_model
        else:
            merged = merged.merge(
                df_model,
                on="variant",
                how=merge_how,
                suffixes=("", "_dup"),
            )

            for col in [
                "num_mutations",
                "sequence",
                "experimental_score",
                "DMS_score",
                "mutant",
                "mutated_sequence",
            ]:
                dup_col = f"{col}_dup"

                if dup_col in merged.columns:
                    merged[col] = merged[col].combine_first(merged[dup_col])
                    merged = merged.drop(columns=[dup_col])

    meta_cols = [
        "variant",
        "num_mutations",
        "sequence",
        "mutated_sequence",
        "mutant",
        "experimental_score",
    ]

    model_cols = [m for m in ACTIVE_MODELS if m in merged.columns]
    other_cols = [
        c for c in merged.columns
        if c not in meta_cols and c not in model_cols
    ]

    ordered_cols = [
        c for c in meta_cols if c in merged.columns
    ] + model_cols + other_cols

    merged = merged[ordered_cols]

    return merged


# Apply ensemble to blind datasets
def apply_ensemble_to_blind(df, weights, active_models, model_signs):
    df = df.copy()

    df = add_oriented_model_columns(df, active_models, model_signs)
    df = add_rank_columns(df, active_models)

    rank_cols = [f"{m}_rank_norm" for m in active_models]

    missing = [c for c in rank_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Blind dataset is missing rank columns after preparation: {missing}")

    X = df[rank_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

    complete = np.all(np.isfinite(X), axis=1)

    df["ensemble_rank_norm"] = np.nan
    df.loc[complete, "ensemble_rank_norm"] = X[complete] @ weights

    # Higher score = better prediction
    df["ensemble_score"] = -df["ensemble_rank_norm"]

    # 1 = best variant
    valid = df["ensemble_score"].notna() & np.isfinite(df["ensemble_score"])

    df["ensemble_rank"] = np.nan
    df.loc[valid, "ensemble_rank"] = df.loc[valid, "ensemble_score"].rank(
        method="average",
        ascending=False,
    )

    return df

# Adjust ensemble for blind DLG4
# Rosetta is not considered 
# Its optimized weight is transferred to ESM2

def transfer_weight_to_model(
    active_models,
    weights,
    source_model="ddg",
    target_model="ESM2",
):
    active_models = list(active_models)
    weights = np.asarray(weights, dtype=float).copy()

    if source_model not in active_models:
        print(f"[INFO] Source model '{source_model}' not in ACTIVE_MODELS. Nothing to transfer.")
        return active_models, weights

    if target_model not in active_models:
        raise ValueError(
            f"Target model '{target_model}' is not in ACTIVE_MODELS. "
            f"Available models: {active_models}"
        )

    source_idx = active_models.index(source_model)
    target_idx = active_models.index(target_model)

    print(f"Transferring weight from {source_model} to {target_model}:")
    print(f"  {source_model} weight: {weights[source_idx]:.6f}")
    print(f"  {target_model} old weight: {weights[target_idx]:.6f}")

    weights[target_idx] += weights[source_idx]

    active_models.pop(source_idx)
    weights = np.delete(weights, source_idx)

    print(f"  {target_model} new weight: {weights[active_models.index(target_model)]:.6f}")
    print(f"  New active models: {active_models}")
    print(f"  New weights sum: {weights.sum():.6f}")

    return active_models, weights


ACTIVE_MODELS_BLIND, BEST_WEIGHTS_BLIND = transfer_weight_to_model(
    active_models=ACTIVE_MODELS,
    weights=BEST_WEIGHTS,
    source_model="ddg",
    target_model="ESM2",
)

# Run
blind_results = {}

for dataset_name, cfg in BLIND_DATASETS.items():
    print(f"\n=== Building merged blind dataset: {dataset_name} ===")

    df_blind = merge_model_predictions(
        dataset_name=dataset_name,
        dataset_cfg=cfg,
        merge_how="outer",
    )

    print(f"{dataset_name}: merged {len(df_blind)} rows")
    print("Available model columns:", available_models_for_df(df_blind, ACTIVE_MODELS))

    df_blind = overwrite_corrected_predictions(df_blind, dataset_name)

    print("Available model columns after overwrites:", available_models_for_df(df_blind, ACTIVE_MODELS))
    
    df_pred = apply_ensemble_to_blind(
        df=df_blind,
        weights=BEST_WEIGHTS_BLIND,
        active_models=ACTIVE_MODELS_BLIND,
        model_signs=MODEL_SIGNS,
    )

    merged_out_path = OUTPUT_DIR / f"{dataset_name}_merged_model_predictions.csv"
    pred_out_path = OUTPUT_DIR / f"{dataset_name}_ensemble_predictions.csv"

    df_blind.to_csv(merged_out_path, index=False)
    df_pred.to_csv(pred_out_path, index=False)

    print(f"Saved merged predictions:   {merged_out_path}")
    print(f"Saved ensemble predictions: {pred_out_path}")
    print(f"NaNs in ensemble_score: {df_pred['ensemble_score'].isna().sum()} / {len(df_pred)}")

    cols_preview = [
        c for c in [
            "variant",
            "mutant",
            "DMS_score",
            "experimental_score",
            "ensemble_score",
            "ensemble_rank",
            "ensemble_rank_norm",
        ]
        if c in df_pred.columns
    ]

    display(df_pred.sort_values("ensemble_rank").head(20)[cols_preview])

    blind_results[dataset_name] = df_pred

Transferring weight from ddg to ESM2:
  ddg weight: 0.000019
  ESM2 old weight: 0.274686
  ESM2 new weight: 0.274705
  New active models: ['ESM2', 'VESPAl', 'prosst', 'esm3_index', 'esmIF_index', 'eve_index']
  New weights sum: 1.000000

=== Building merged blind dataset: dlg4-2022-abundance ===
ESM2: filtered num_mutations == 1: 1280 -> 1280
dlg4-2022-abundance | ESM2: loaded (1280, 5)
esm3_index: filtered num_mutations == 1: 6976 -> 1280
dlg4-2022-abundance | esm3_index: loaded (1280, 5)
esmIF_index: filtered num_mutations == 1: 6976 -> 1280
dlg4-2022-abundance | esmIF_index: loaded (1280, 5)
eve_index: filtered num_mutations == 1: 6976 -> 1280
dlg4-2022-abundance | eve_index: loaded (1280, 5)
prosst: filtered num_mutations == 1: 1280 -> 1280
dlg4-2022-abundance | prosst: loaded (1280, 5)
VESPAl: filtered num_mutations == 1: 1280 -> 1280
dlg4-2022-abundance | VESPAl: loaded (1280, 5)
dlg4-2022-abundance: merged 1280 rows
Available model columns: ['ESM2', 'VESPAl', 'prosst', 'esm3_ind

,variant,experimental_score,ensemble_score,ensemble_rank,ensemble_rank_norm
601,I3V,-0.067188,-0.030170,1.0,0.030170
9,A32P,0.003088,-0.031406,2.0,0.031406
677,I78V,0.067664,-0.031435,3.0,0.031435
642,I66L,-0.083287,-0.036935,4.0,0.036935
717,K82R,0.092358,-0.037563,5.0,0.037563
689,K44R,-0.359379,-0.037575,6.0,0.037575
1121,S60T,0.074104,-0.041778,7.0,0.041778
670,I78L,0.027543,-0.042466,8.0,0.042466
117,D21E,0.077612,-0.047570,9.0,0.047570
870,N58S,-0.039764,-0.048095,10.0,0.048095



=== Building merged blind dataset: dlg4-2022-binding ===
ESM2: filtered num_mutations == 1: 1441 -> 1441
dlg4-2022-binding | ESM2: loaded (1441, 5)
esm3_index: filtered num_mutations == 1: 8251 -> 1441
dlg4-2022-binding | esm3_index: loaded (1441, 5)
esmIF_index: filtered num_mutations == 1: 8251 -> 1441
dlg4-2022-binding | esmIF_index: loaded (1441, 5)
eve_index: filtered num_mutations == 1: 8251 -> 1441
dlg4-2022-binding | eve_index: loaded (1441, 5)
prosst: filtered num_mutations == 1: 1441 -> 1441
dlg4-2022-binding | prosst: loaded (1441, 5)
VESPAl: filtered num_mutations == 1: 1441 -> 1441
dlg4-2022-binding | VESPAl: loaded (1441, 5)
dlg4-2022-binding: merged 1441 rows
Available model columns: ['ESM2', 'VESPAl', 'prosst', 'esm3_index', 'esmIF_index', 'eve_index']
[dlg4-2022-binding] no corrected VESPA file found; keeping existing VESPAl if present.
[dlg4-2022-binding] no corrected ProSST file found; keeping existing prosst if present.
Available model columns after overwrites: ['E

,variant,experimental_score,ensemble_score,ensemble_rank,ensemble_rank_norm
759,I78V,0.007666,-0.031669,1.0,0.031669
673,I3V,-0.099851,-0.031697,2.0,0.031697
9,A32P,0.159133,-0.031856,3.0,0.031856
718,I66L,0.072435,-0.037412,4.0,0.037412
805,K82R,0.174476,-0.038167,5.0,0.038167
773,K44R,-0.149268,-0.039242,6.0,0.039242
725,I66V,0.051398,-0.039256,7.0,0.039256
1130,R2K,0.058923,-0.039256,8.0,0.039256
1257,S60T,-0.195528,-0.042248,9.0,0.042248
752,I78L,-0.001334,-0.044348,10.0,0.044348


In [ ]:
# Evaluate DLG4 blind ensemble predictions

POSSIBLE_LABEL_COLS = [
    "experimental_score",
    "score",
    "DMS_score",
]

DLG4_EVAL_OUTPUT_DIR = OUTPUT_DIR
DLG4_EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

dlg4_ensemble_eval_rows = []

for dataset_name, df in blind_results.items():
    print(f"\n=== Evaluating {dataset_name} ===")

    if "ensemble_score" not in df.columns:
        print(f"[SKIP] {dataset_name}: ensemble_score not found.")
        continue

    label_cols = [c for c in POSSIBLE_LABEL_COLS if c in df.columns]

    if not label_cols:
        print(f"[SKIP] {dataset_name}: no experimental label column found.")
        print("Available columns:", df.columns.tolist())
        continue

    for label_col in label_cols:
        y_true = numeric_array(df[label_col])
        y_pred = numeric_array(df["ensemble_score"])

        metrics = summarize_prediction(
            true_values=y_true,
            pred_scores=y_pred,
            fracs=OPTIMIZATION_FRACS,
        )

        dlg4_ensemble_eval_rows.append({
            "dataset": dataset_name,
            "target": label_col,
            "model": "optimized_ensemble",
            **metrics,
        })

dlg4_ensemble_eval_df = pd.DataFrame(dlg4_ensemble_eval_rows)

if len(dlg4_ensemble_eval_df) > 0:
    display(dlg4_ensemble_eval_df.round(4))

    out_path = DLG4_EVAL_OUTPUT_DIR / "dlg4_blind_ensemble_metrics.csv"
    dlg4_ensemble_eval_df.to_csv(out_path, index=False)

    print(f"\nSaved ensemble metrics to: {out_path}")
else:
    print("No DLG4 ensemble metrics computed.")


=== Evaluating dlg4-2022-abundance ===

=== Evaluating dlg4-2022-binding ===


,dataset,target,model,n,pearson,spearman,ndcg@5%,ndcg@10%,ndcg@20%
0,dlg4-2022-abundance,experimental_score,optimized_ensemble,1280,0.7207,0.7063,0.8723,0.9015,0.9142
1,dlg4-2022-binding,experimental_score,optimized_ensemble,1441,0.7957,0.7855,0.8935,0.9123,0.9315



Saved ensemble metrics to: /Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/ensemble_outputs/dlg4_blind/dlg4_blind_ensemble_metrics.csv


In [ ]:
# Evaluate DLG4 blind ensemble and individual zero-shot models

POSSIBLE_LABEL_COLS = [
    "experimental_score",
    "score",
    "DMS_score",
    "binding",
    "abundance",
    "fitness",
]

dlg4_all_model_eval_rows = []

for dataset_name, df in blind_results.items():
    print(f"\n=== Evaluating {dataset_name} ===")

    label_cols = [c for c in POSSIBLE_LABEL_COLS if c in df.columns]

    if not label_cols:
        print(f"[SKIP] {dataset_name}: no experimental label column found.")
        continue

    models_to_evaluate = []

    # Ensemble
    if "ensemble_score" in df.columns:
        models_to_evaluate.append(("optimized_ensemble", "ensemble_score"))

    # Individual models, using oriented columns if available
    for model in ACTIVE_MODELS_BLIND:
        oriented_col = f"{model}_oriented"

        if oriented_col in df.columns:
            models_to_evaluate.append((model, oriented_col))
        elif model in df.columns:
            models_to_evaluate.append((model, model))

    for label_col in label_cols:
        y_true = numeric_array(df[label_col])

        for model_name, pred_col in models_to_evaluate:
            y_pred = numeric_array(df[pred_col])

            metrics = summarize_prediction(
                true_values=y_true,
                pred_scores=y_pred,
                fracs=OPTIMIZATION_FRACS,
            )

            dlg4_all_model_eval_rows.append({
                "dataset": dataset_name,
                "target": label_col,
                "model": model_name,
                "prediction_column": pred_col,
                **metrics,
            })

dlg4_all_model_eval_df = pd.DataFrame(dlg4_all_model_eval_rows)

if len(dlg4_all_model_eval_df) > 0:
    display(
        dlg4_all_model_eval_df
        .sort_values(["dataset", "target", "ndcg@10%"], ascending=[True, True, False])
        .round(4)
    )

    out_path = OUTPUT_DIR / "dlg4_blind_all_model_metrics.csv"
    dlg4_all_model_eval_df.to_csv(out_path, index=False)

    print(f"\nSaved all model metrics to: {out_path}")
else:
    print("No DLG4 metrics computed.")


=== Evaluating dlg4-2022-abundance ===

=== Evaluating dlg4-2022-binding ===


,dataset,target,model,prediction_column,n,pearson,spearman,ndcg@5%,ndcg@10%,ndcg@20%
4,dlg4-2022-abundance,experimental_score,esm3_index,esm3_index_oriented,1280,0.6733,0.6613,0.8999,0.9079,0.9147
3,dlg4-2022-abundance,experimental_score,prosst,prosst_oriented,1280,0.7564,0.7460,0.8932,0.9072,0.9280
0,dlg4-2022-abundance,experimental_score,optimized_ensemble,ensemble_score,1280,0.7207,0.7063,0.8723,0.9015,0.9142
5,dlg4-2022-abundance,experimental_score,esmIF_index,esmIF_index_oriented,1280,0.6654,0.6639,0.8894,0.8969,0.9146
2,dlg4-2022-abundance,experimental_score,VESPAl,VESPAl_oriented,1280,0.5253,0.4998,0.8820,0.8865,0.8775
1,dlg4-2022-abundance,experimental_score,ESM2,ESM2_oriented,1280,0.5276,0.5183,0.8572,0.8664,0.8606
6,dlg4-2022-abundance,experimental_score,eve_index,eve_index_oriented,1280,0.3266,0.3118,0.8314,0.8439,0.8490
7,dlg4-2022-binding,experimental_score,optimized_ensemble,ensemble_score,1441,0.7957,0.7855,0.8935,0.9123,0.9315
9,dlg4-2022-binding,experimental_score,VESPAl,VESPAl_oriented,1441,0.6295,0.6321,0.8763,0.8955,0.9061
8,dlg4-2022-binding,experimental_score,ESM2,ESM2_oriented,1441,0.6473,0.6510,0.8708,0.8936,0.9001



Saved all model metrics to: /Users/kevinbauer/Desktop/Masterarbeit/Zero-Shot_Phase/Final/ensemble_outputs/dlg4_blind/dlg4_blind_all_model_metrics.csv
